# Bài 6: Schema Enforcement, Schema Evolution & Constraints

## Mục tiêu
- Hiểu Delta Lake **từ chối** ghi dữ liệu sai schema theo mặc định (schema enforcement).
- Cho phép mở rộng schema có kiểm soát bằng `mergeSchema` / `autoMerge`.
- Dùng `ALTER TABLE` để thêm/sửa/đổi tên cột.
- Thêm ràng buộc dữ liệu: `NOT NULL`, `CHECK constraint`, và **generated column**.


## 6.1. Schema Enforcement

Mặc định, khi ghi (`append`) vào bảng đã tồn tại, Delta **so khớp schema** của DataFrame với schema hiện tại của bảng:
- Thiếu cột, thừa cột, sai kiểu (không tự động cast an toàn được) → **ném lỗi ngay**, không ghi gì cả.
- Đây là điểm khác biệt lớn với Parquet/Hive thuần (thường "âm thầm" tạo ra bảng có schema không đồng nhất giữa các file).

## 6.2. Schema Evolution có kiểm soát

Muốn cho phép DataFrame có **thêm cột mới** khi ghi, phải khai báo tường minh:

```python
df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("db.tbl")
```
hoặc bật ở session level: `spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")`.

Với `overwrite`, muốn **thay hẳn schema** (không chỉ thêm cột) dùng:
```python
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("db.tbl")
```

`mergeSchema` chỉ cho phép các thay đổi **an toàn** (thêm cột mới ở cuối, nới kiểu số...); đổi tên/xoá cột hay đổi kiểu không tương thích vẫn phải dùng `ALTER TABLE` tường minh hoặc `overwriteSchema`.

## 6.3. `ALTER TABLE`

```sql
ALTER TABLE db.tbl ADD COLUMNS (email STRING);
ALTER TABLE db.tbl ALTER COLUMN price TYPE DOUBLE;
ALTER TABLE db.tbl RENAME COLUMN name TO full_name;     -- can column mapping mode = name
ALTER TABLE db.tbl DROP COLUMN old_col;                 -- can column mapping mode = name
```
`RENAME COLUMN`/`DROP COLUMN` yêu cầu bật **column mapping**:
```sql
ALTER TABLE db.tbl SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
```

## 6.4. Constraints

```sql
ALTER TABLE db.tbl ADD CONSTRAINT positive_price CHECK (price > 0);
ALTER TABLE db.tbl CHANGE COLUMN id SET NOT NULL;
```
Nếu có bất kỳ dòng nào vi phạm constraint đã tồn tại trong bảng, lệnh `ADD CONSTRAINT` sẽ thất bại ngay. Sau khi thêm, mọi `INSERT`/`UPDATE`/`MERGE` vi phạm sẽ bị từ chối.

## 6.5. Generated columns

Cột được **tự động tính** từ các cột khác, dùng tiện lợi để tạo cột partition từ timestamp mà không cần app ghi thủ công:

```sql
CREATE TABLE db.events (
    event_time TIMESTAMP,
    event_date DATE GENERATED ALWAYS AS (CAST(event_time AS DATE))
) USING DELTA
PARTITIONED BY (event_date);
```
Khi ghi, không cần cung cấp `event_date` — Delta tự tính. Nếu có cung cấp mà giá trị sai với công thức, Delta sẽ báo lỗi.


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai06"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai06-schema")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 6.6. Ví dụ minh hoạ

In [ ]:
spark.sql("DROP TABLE IF EXISTS bai06.users")
spark.createDataFrame([(1,"An"),(2,"Binh")], ["id","name"]).write.format("delta").saveAsTable("bai06.users")

# Thu ghi them 1 cot moi khong khai bao mergeSchema -> loi
try:
    spark.createDataFrame([(3,"Chi","chi@mail.com")], ["id","name","email"]) \
        .write.format("delta").mode("append").saveAsTable("bai06.users")
except Exception as e:
    print("Loi nhu du kien (schema mismatch):", type(e).__name__)


In [ ]:
# Cho phep mo rong schema
spark.createDataFrame([(3,"Chi","chi@mail.com")], ["id","name","email"]) \
    .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("bai06.users")

spark.sql("SELECT * FROM bai06.users ORDER BY id").show()
# id=1,2 co email = NULL vi duoc them truoc khi cot email ton tai


In [ ]:
# ALTER TABLE: bat column mapping, doi ten cot, them constraint
spark.sql("ALTER TABLE bai06.users SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")
spark.sql("ALTER TABLE bai06.users RENAME COLUMN name TO full_name")
spark.sql("ALTER TABLE bai06.users ADD CONSTRAINT id_not_null CHECK (id IS NOT NULL)")

spark.sql("DESCRIBE TABLE bai06.users").show(truncate=False)


In [ ]:
# Vi pham constraint -> bi tu choi
try:
    spark.sql("INSERT INTO bai06.users (id, full_name, email) VALUES (NULL, 'X', 'x@mail.com')")
except Exception as e:
    print("Loi nhu du kien (constraint violation):", type(e).__name__)


In [ ]:
# Generated column: tu dong tinh event_date tu event_time, dung lam cot partition
spark.sql("DROP TABLE IF EXISTS bai06.events")
spark.sql("""
CREATE TABLE bai06.events (
    event_id INT,
    event_time TIMESTAMP,
    event_date DATE GENERATED ALWAYS AS (CAST(event_time AS DATE))
) USING DELTA
PARTITIONED BY (event_date)
""")
spark.sql("""
INSERT INTO bai06.events (event_id, event_time) VALUES
  (1, TIMESTAMP'2024-05-01 10:00:00'),
  (2, TIMESTAMP'2024-05-02 08:30:00')
""")
spark.sql("SELECT * FROM bai06.events ORDER BY event_id").show()
spark.sql("SHOW PARTITIONS bai06.events").show()


## 6.7. Thực hành

**Bài 1** — Tạo bảng `bai06.products (id INT, name STRING, price DOUBLE)`. Thử `append` 1 DataFrame thêm cột `category` mà **không** dùng `mergeSchema` — quan sát lỗi.

**Bài 2** — Ghi lại đúng thao tác ở Bài 1 nhưng với `option("mergeSchema", "true")`. Kiểm tra các dòng cũ có giá trị `category` là gì.

**Bài 3** — Bật `column mapping` cho `bai06.products`, đổi tên cột `price` thành `unit_price` bằng `ALTER TABLE ... RENAME COLUMN`.

**Bài 4** — Thêm constraint để `unit_price` luôn `> 0`. Thử `INSERT` 1 dòng vi phạm để xác nhận bị chặn, rồi `INSERT` 1 dòng hợp lệ để xác nhận vẫn hoạt động bình thường.

**Bài 5** — Tạo bảng mới `bai06.orders` có cột `order_ts TIMESTAMP` và 1 generated column `order_month STRING GENERATED ALWAYS AS (date_format(order_ts, 'yyyy-MM'))`, partition theo `order_month`. Insert vài dòng với `order_ts` khác tháng nhau và kiểm tra `SHOW PARTITIONS`.


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

In [ ]:
# TODO: Bài 4


### Vùng làm bài — Bài 5

In [ ]:
# TODO: Bài 5


---
## Gợi ý / đáp án tham khảo

In [ ]:
# Dap an Bai 1
spark.sql("DROP TABLE IF EXISTS bai06.products")
spark.createDataFrame([(1,"Keyboard",25.0),(2,"Mouse",10.0)], ["id","name","price"]) \
    .write.format("delta").saveAsTable("bai06.products")
try:
    spark.createDataFrame([(3,"Monitor",199.0,"Electronics")], ["id","name","price","category"]) \
        .write.format("delta").mode("append").saveAsTable("bai06.products")
except Exception as e:
    print("Loi nhu du kien:", type(e).__name__)


In [ ]:
# Dap an Bai 2
spark.createDataFrame([(3,"Monitor",199.0,"Electronics")], ["id","name","price","category"]) \
    .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("bai06.products")
spark.sql("SELECT * FROM bai06.products ORDER BY id").show()
# id=1,2 co category = NULL


In [ ]:
# Dap an Bai 3
spark.sql("ALTER TABLE bai06.products SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")
spark.sql("ALTER TABLE bai06.products RENAME COLUMN price TO unit_price")
spark.sql("DESCRIBE TABLE bai06.products").show(truncate=False)


In [ ]:
# Dap an Bai 4
spark.sql("ALTER TABLE bai06.products ADD CONSTRAINT positive_price CHECK (unit_price > 0)")
try:
    spark.sql("INSERT INTO bai06.products (id, name, unit_price, category) VALUES (4, 'Bad', -5.0, 'X')")
except Exception as e:
    print("Loi nhu du kien:", type(e).__name__)
spark.sql("INSERT INTO bai06.products (id, name, unit_price, category) VALUES (5, 'Good', 15.0, 'X')")
spark.sql("SELECT * FROM bai06.products ORDER BY id").show()


In [ ]:
# Dap an Bai 5
spark.sql("DROP TABLE IF EXISTS bai06.orders")
spark.sql("""
CREATE TABLE bai06.orders (
    order_id INT,
    order_ts TIMESTAMP,
    order_month STRING GENERATED ALWAYS AS (date_format(order_ts, 'yyyy-MM'))
) USING DELTA
PARTITIONED BY (order_month)
""")
spark.sql("""
INSERT INTO bai06.orders (order_id, order_ts) VALUES
  (1, TIMESTAMP'2024-01-15 10:00:00'),
  (2, TIMESTAMP'2024-02-20 11:00:00')
""")
spark.sql("SELECT * FROM bai06.orders ORDER BY order_id").show()
spark.sql("SHOW PARTITIONS bai06.orders").show()
